## Importing libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm
from pytorch_datasets.augmentations import Cutout

Notes:
* AugReg is powerful. performance as x10 data.
* Transfer is better, but AugReg is useless. Sometimes impossible to reach transfer perf with training from scratch.
* More data -> more general models.  using more data with a fixed compute budget (more data with less compute time is better that less data with more compute time)
* Aug is better than reg. For 21k no augreg is best.
* Model can be selected by validating on a upstream val data instead of adapting each model and then validating it (mostly as good).
* Bigger patch size rather than smaller model

## Initial Parameters

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Loading Data

### Selecting Transformations

In [3]:
mean    = [0.485, 0.456, 0.406]
std     = [0.229, 0.224, 0.225]

aug_l   = 2
mag     = 15
alpha   = 0.2

batch_size = 64
# Set up data transformations
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(magnitude=mag, num_ops=aug_l),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

### Loading Datasets

In [9]:
# Load CIFAR-10 dataset
# trainset = datasets.ImageNet(root='E:\Datasets\ImageNet1k', split='train',  transform=transform)
valset = datasets.ImageNet(root='E:\Datasets\ImageNet1k', split='val',  transform=transform)


# train_loader = DataLoader(
#     trainset, 
#     batch_size=batch_size, 
#     shuffle=True,
#     num_workers=4, 
#     pin_memory=True,
#     persistent_workers=True
# )

val_loader = DataLoader(
    valset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True,
    persistent_workers=True
)

## Selecting a model

### Loading a model

In [18]:
# timm.list_models()

In [19]:
pretrained = True
# Define the model
# model = timm.create_model("resnet50", pretrained=pretrained, num_classes=10)
model = timm.create_model("vit_base_patch32_224_in21k", pretrained=pretrained, num_classes=100)

In [20]:
# for parameter in model.parameters():
#     parameter.requires_grad = False
    
# for parameter in model.head.parameters():
#     parameter.requires_grad = True


In [21]:
class SE(nn.Module):
    def __init__(self, in_channels, reduction_ratio=4):
        super(SE, self).__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction_ratio)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_channels // reduction_ratio, in_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return x * out

### Customizing the model

In [22]:
# new_head = nn.Sequential(SE(192),
#                          model.head)

# model.head = new_head

In [23]:
model = model.to(device)

## Training

### Training Parameters

In [24]:
num_epochs = 5
steps = num_epochs * 391

In [25]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR
# Define loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-3, weight_decay=0.3)

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.02,
    end_factor=1, 
    total_iters=5
)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=steps-5,
    eta_min=1e-5
)

seq_scheduler = SequentialLR(
    optimizer, 
    [warmup_scheduler, cosine_scheduler],
    milestones=[5]
)
# scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

scheduler = seq_scheduler

### Training loop

In [26]:
best_model = None
best_val_loss = float('inf')
# scaler = GradScaler()

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    
    for inputs, labels in tqdm(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        # Update the learning rate scheduler
        scheduler.step()

        train_loss += loss.item() * inputs.size(0)
        train_correct += torch.sum(torch.argmax(outputs, dim=1) == labels)
        # break

    train_loss /= len(train_loader.dataset)
    train_accuracy = train_correct.float() / len(train_loader.dataset)


    # Evaluation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_correct += torch.sum(torch.argmax(outputs, dim=1) == labels)

        val_loss /= len(val_loader.dataset)
        val_accuracy = val_correct.float() / len(val_loader.dataset)

    print('Epoch [{}/{}], Train Loss: {:.4f}, Train Acc: {:.4f}, Val Loss: {:.4f}, Val Acc: {:.4f}'.format(
        epoch + 1, num_epochs, train_loss, train_accuracy, val_loss, val_accuracy))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model.state_dict()
        print('!!! New best model found !!!')
        # torch.save(best_model, 'best_model.pth')

  0%|          | 0/391 [00:08<?, ?it/s]

## Save the model

In [ ]:
# torch.save(best_model, 'resnet_best_model.pth')